# W5D4 — Attention and the Transformer Block — Guided

**Week 5 · Day 4 · NLP Foundations** · Lab

Two kinds of memory so far. The RNN carried a state from token to token, and its memory of the first
token faded with every step. The TCN looked through a fixed window and saw nothing outside it.

Attention removes both limits. Every token scores every other token, turns the scores into weights
with a softmax, and takes a weighted sum — **one step, any distance**. You compute it by hand on three
vectors — the same three the RNN read on Tuesday — and then you prove that, on its own, it is
completely blind to order. Then you add what fixes that, build a full transformer block around it, and
put it through the memory task that the RNN could not reliably pass.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٥ · اليوم ٤ — الانتباه وكتلة المحوّل

**الأسبوع الخامس · اليوم الرابع · أساسيات معالجة اللغة** · معمل

نوعان من الذاكرة حتى الآن. حملت RNN حالةً من رمز إلى رمز، فبهتت ذاكرتها للرمز الأول مع كل خطوة. ونظرت TCN عبر
نافذة ثابتة فلم ترَ شيئًا خارجها.

والانتباه يرفع القيدين كليهما. كل رمز يُقيّم كل رمز آخر، ويحوّل الدرجات إلى أوزان بـsoftmax، ويأخذ مجموعًا موزونًا
— **خطوة واحدة، وأي مسافة**. تحسبه يدويًا على ثلاثة متّجهات — الثلاثة نفسها التي قرأتها RNN يوم الثلاثاء — ثم
تُبرهن أنه وحده أعمى تمامًا عن الترتيب. ثم تضيف ما يُصلح ذلك، وتبني حوله كتلة محوّل كاملة، وتُدخلها مهمة الذاكرة
التي لم تجتزها RNN باطّراد.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Compute attention by hand — scores, softmax, weighted sum — and reproduce the slide's numbers.
- Prove that attention without positions cannot see order, and show positional encoding fixing it.
- Scale scores by `√d` and say what happens without it.
- Build multi-head attention from scratch and match `nn.MultiheadAttention` to `1e-5`.
- Assemble a transformer block with a shape check at every step, and apply a causal mask.
- Show a tiny transformer solving a memory task the RNN could not reliably pass.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تحسب الانتباه يدويًا — الدرجات وsoftmax والمجموع الموزون — وتُعيد إنتاج أرقام الشريحة.
- أن تُبرهن أن الانتباه بلا مواضع لا يرى الترتيب، وتُظهر ترميز الموضع يُصلحه.
- أن تقسم الدرجات على `√d` وتقول ما يحدث بدون ذلك.
- أن تبني الانتباه متعدّد الرؤوس من الصفر وتطابقه مع `nn.MultiheadAttention` إلى `1e-5`.
- أن تُركّب كتلة محوّل مع فحص للشكل في كل خطوة، وتطبّق قناعًا سببيًا.
- أن تُظهر محوّلًا صغيرًا يحلّ مهمة ذاكرة لم تجتزها RNN باطّراد.

</div>

## About the data

The warm-up uses **three hard-coded vectors**, on purpose: no data loading between you and the
mechanism. The core uses the memory task — label = first token, chance 0.25 — and its table so far,
read from `memory_task_D2.json`.

The stretch trains a tiny transformer on `reviews_sentiment`, on the week's split (`week5_split.json`,
hash checked), tokenised exactly as on Tuesday, and adds it to the leaderboard.

Nothing downloads today. The slowest cell, task 2.5, trains one tiny transformer for about half a
minute; the stretch trains another for about two.

<div dir="rtl" align="right">

## عن البيانات

يستعمل الإحماء **ثلاثة متّجهات مكتوبة في الشيفرة**، عمدًا: لا تحميل بيانات بينك وبين الآلية. ويستعمل الأساسي
مهمة الذاكرة — التسمية هي الرمز الأول، والصدفة ٠٫٢٥ — وجدولها حتى الآن من `memory_task_D2.json`.

ويُدرّب القسم الإضافي محوّلًا صغيرًا على `reviews_sentiment`، على تقسيم الأسبوع (`week5_split.json` مع التحقّق
من البصمة)، مُرمَّزةً تمامًا كيوم الثلاثاء، ويضيفه إلى اللوحة.

لا يُنزَّل شيء اليوم. وأبطأ خلية، المهمة ٢٫٥، تُدرّب محوّلًا صغيرًا نحو نصف دقيقة؛ ويُدرّب القسم الإضافي آخر نحو
دقيقتين.

</div>

## Setup

<div dir="rtl" align="right">

## الإعداد

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import get_dataset, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, report

ensure("matplotlib")
seed_everything(42)

import collections
import hashlib
import json
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEED = 42
torch.set_num_threads(4)
np.set_printoptions(precision=4, suppress=True)

# The three tokens from this morning's slide — and from Tuesday's RNN — and the query.
V = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0]])
QUERY = np.array([1.0, 0.0])
print(versions())

## Section 1 — Warm-up: attention, by hand  (≈25 min)

Everything here works.

The slide's example, step by step:

1. Three scores, one dot product each: `q · v₁ = 1`, `q · v₂ = 0`, `q · v₃ = 1`.
2. Softmax: `e¹, e⁰, e¹` = 2.72, 1.00, 2.72, sum 6.44 → weights **0.42, 0.16, 0.42**.
3. Weighted sum: `0.42·[1,0] + 0.16·[0,1] + 0.42·[1,1]` = **`[0.84, 0.58]`**.

The softmax subtracts the largest score first. That changes nothing mathematically and keeps
`exp` from overflowing on large scores — try `[1000, 1001]` without it.

Then change the query to `[0, 1]`: the weights flip to **0.16, 0.42, 0.42**. The same tokens, read
differently, because the question changed.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: الانتباه، يدويًا (نحو ٢٥ دقيقة)

كل ما هنا يعمل.

مثال الشريحة خطوةً خطوة:

١. ثلاث درجات، جداء نقطي لكلٍّ منها: `q · v₁ = 1` و`q · v₂ = 0` و`q · v₃ = 1`.
٢. softmax: `e¹, e⁰, e¹` = ٢٫٧٢ و١٫٠٠ و٢٫٧٢، ومجموعها ٦٫٤٤ ← الأوزان **٠٫٤٢ و٠٫١٦ و٠٫٤٢**.
٣. المجموع الموزون: `0.42·[1,0] + 0.16·[0,1] + 0.42·[1,1]` = **`[0.84, 0.58]`**.

تطرح softmax أكبر درجة أولًا. وهذا لا يغيّر شيئًا رياضيًا ويمنع `exp` من الطفح على الدرجات الكبيرة — جرّب
`[1000, 1001]` بدونه.

ثم غيّر المُستعلِم إلى `[0, 1]`: فتنقلب الأوزان إلى **٠٫١٦ و٠٫٤٢ و٠٫٤٢**. الرموز نفسها تُقرأ بطريقة مختلفة،
لأن السؤال تغيّر.

</div>

In [ ]:
def softmax(scores, axis=-1):
    """Numerically stable softmax: subtract the max, exponentiate, normalise."""
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=axis, keepdims=True)


def attention(q, values):
    """Single-query attention. Returns (output, weights)."""
    weights = softmax(values @ q)
    return weights @ values, weights


OUTPUT, WEIGHTS = attention(QUERY, V)
FLIPPED_OUTPUT, FLIPPED_WEIGHTS = attention(np.array([0.0, 1.0]), V)
print(f"scores  : {V @ QUERY}")
print(f"weights : {WEIGHTS.round(2)}        output {OUTPUT.round(2)}")
print(f"flipped : {FLIPPED_WEIGHTS.round(2)}        output {FLIPPED_OUTPUT.round(2)}")
with np.errstate(over="ignore", invalid="ignore"):
    print(f"\n[1000, 1001] naive : {np.exp([1000.0, 1001.0]) / np.exp([1000.0, 1001.0]).sum()}")
print(f"[1000, 1001] stable: {softmax(np.array([1000.0, 1001.0])).round(4)}")

### Attention is blind to order

Feed the same three tokens in **reverse** order with the same query. On Tuesday the RNN's final state
changed when you did this. Here the output is **bit-for-bit identical**: each weight depends only on
its own token, and a sum does not care about the order of its terms. Position is nowhere in the
arithmetic. That is the first thing a transformer has to add.

<div dir="rtl" align="right">

### الانتباه أعمى عن الترتيب

أدخِل الرموز الثلاثة نفسها بترتيب **معكوس** بالمُستعلِم نفسه. تغيّرت الحالة الأخيرة لـRNN يوم الثلاثاء حين
فعلت هذا. أمّا هنا فالمخرَج **متطابق بتًّا ببت**: فكل وزن يعتمد على رمزه وحده، والمجموع لا يبالي بترتيب حدوده.
والموضع ليس في الحساب في أي مكان. وهذا أول ما يجب أن يضيفه المحوّل.

</div>

In [ ]:
REVERSED_OUTPUT, REVERSED_WEIGHTS = attention(QUERY, V[::-1])
ORDER_BLIND = np.array_equal(OUTPUT, REVERSED_OUTPUT)
print(f"in order : weights {WEIGHTS.round(4)} -> output {OUTPUT}")
print(f"reversed : weights {REVERSED_WEIGHTS.round(4)} -> output {REVERSED_OUTPUT}")
print(f"identical output: {ORDER_BLIND}")

## Section 2 — Core: five tasks  (≈60 min)

1. Scaled dot-product attention on matrices, and what `√d` is for.
2. Positional encoding: give attention a sense of order.
3. Multi-head attention from scratch, matched against PyTorch.
4. The transformer block, with a shape check at every step — and the causal mask.
5. The memory task, second round.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: خمس مهام (نحو ٦٠ دقيقة)

١. انتباه الجداء النقطي المقيس على المصفوفات، وفيمَ يُستعمل `√d`.
٢. ترميز الموضع: امنح الانتباه حسًّا بالترتيب.
٣. الانتباه متعدّد الرؤوس من الصفر، مطابقًا لـPyTorch.
٤. كتلة المحوّل، مع فحص للشكل في كل خطوة — والقناع السببي.
٥. مهمة الذاكرة، الجولة الثانية.

</div>

### Task 2.1 — scaled dot-product attention

In a real model every token asks its own question, so the queries form a matrix `Q`, the things they
are matched against form `K`, and the things they retrieve form `V`:

`attention(Q, K, V) = softmax(Q Kᵀ / √d) V`

Write it in PyTorch, softmax over the **last** axis (each query's row sums to 1), and run it with the
slide's vectors as `K` and `V` and two queries at once. Row 0 is the warm-up again, softened a little
by the division by `√2`.

Then the `√d`. A dot product of two random `d`-dimensional vectors grows like `√d`, and a softmax of
large numbers is close to one-hot — all the weight on one key, and almost no gradient to the others. For
`d = 64`, compare one query's weights over eight random keys with and without the division.

<div dir="rtl" align="right">

### المهمة ٢٫١ — انتباه الجداء النقطي المقيس

في نموذج حقيقي يسأل كل رمز سؤاله، فتكوّن المُستعلِمات مصفوفة `Q`، وما تُطابَق عليه `K`، وما تسترجعه `V`:

`attention(Q, K, V) = softmax(Q Kᵀ / √d) V`

اكتبها في PyTorch، وطبّق softmax على المحور **الأخير** (يجمع صفّ كل مُستعلِم إلى ١)، وشغّلها ومتّجهات الشريحة
هي `K` و`V` ومعها مُستعلِمان دفعةً واحدة. فالصف ٠ هو الإحماء مرّة أخرى، وقد ليّنته القسمة على `√2` قليلًا.

ثم `√d`. الجداء النقطي لمتّجهين عشوائيين بُعدهما `d` يكبر مثل `√d`، وsoftmax الأعداد الكبيرة قريبة من الأحادية
الساخنة (one-hot) — الوزن كله على مفتاح واحد، ولا تدرّج تقريبًا إلى البقية. ولـ`d = 64`، قارن أوزان مُستعلِم واحد على
ثمانية مفاتيح عشوائية بالقسمة وبدونها.

</div>

In [ ]:
def scaled_attention(Q, K, V, mask=None):
    """softmax(Q K^T / sqrt(d)) V over the last two axes. Returns (output, weights)."""
    # TODO: Score every query against every key, divide by sqrt(d), mask (fill with -inf) if given, softmax over the last axis, and mix the values.
    # مهمة: قيّم كل مُستعلِم مقابل كل مفتاح، واقسم على `sqrt(d)`، وطبّق القناع (املأ بـ`-inf`) إن أُعطي، وsoftmax على المحور الأخير، واخلط القيم.


slide = torch.tensor(V, dtype=torch.float32)
queries = torch.tensor([[1.0, 0.0], [0.0, 1.0]])               # the warm-up's query and the flipped one
_, SLIDE_WEIGHTS = scaled_attention(queries, slide, slide)
SLIDE_MATCH = np.allclose(SLIDE_WEIGHTS[0].numpy(), softmax(V @ QUERY / np.sqrt(2)), atol=1e-6)
print("weights, one row per query:")
print(SLIDE_WEIGHTS.numpy().round(2))
print(f"row 0 = the warm-up's softmax of [1, 0, 1] / sqrt(2): {SLIDE_MATCH} — the division softens 0.42/0.16/0.42")

In [ ]:
torch.manual_seed(SEED)
q, keys = torch.randn(1, 64), torch.randn(8, 64)
raw = (q @ keys.T).flatten()
UNSCALED, SCALED = torch.softmax(raw, 0), torch.softmax(raw / 64 ** 0.5, 0)
print(f"scores range {raw.min():.1f} … {raw.max():.1f}")
print(f"max weight without / sqrt(d): {UNSCALED.max():.3f}   with: {SCALED.max():.3f}")
plt.bar(np.arange(8) - 0.2, UNSCALED, width=0.4, label="unscaled")
plt.bar(np.arange(8) + 0.2, SCALED, width=0.4, label="÷ √64")
plt.title("one query over eight keys, d = 64")
plt.legend()
plt.show()

### Task 2.2 — positional encoding

Attention cannot see order, so give every position its own vector and **add** it to the token vector
before anything else. The sinusoidal encoding from the paper:

`PE(pos, 2i) = sin(pos / 10000^(2i/d))`, `PE(pos, 2i+1) = cos(pos / 10000^(2i/d))`

Each dimension is a wave of a different wavelength, so every position gets a different pattern. It is a
function of the position, not something learned, so position 5 is the same vector however long the
sequence. Plot it as a heatmap.

Then redo the warm-up with positions. Add the slide's small positional vectors `[0.1, 0]`,
`[0, 0.1]`, `[0.1, 0.1]` to the three tokens, swap the first two tokens while leaving the positions in
place, and the output changes. Order is visible now.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — ترميز الموضع

لا يرى الانتباه الترتيب، فامنح كل موضع متّجهه و**أضِفه** إلى متّجه الرمز قبل أي شيء. الترميز الجيبي من الورقة:

`PE(pos, 2i) = sin(pos / 10000^(2i/d))`، `PE(pos, 2i+1) = cos(pos / 10000^(2i/d))`

كل بُعد موجة بطول موجي مختلف، فيأخذ كل موضع نمطًا مختلفًا. وهو دالّة في الموضع لا شيء مُتعلَّم، فالموضع ٥ هو
المتّجه نفسه مهما طالت المتتالية. ارسمه خريطةً حرارية.

ثم أعِد الإحماء بالمواضع. أضِف متّجهات المواضع الصغيرة من الشريحة `[0.1, 0]` و`[0, 0.1]` و`[0.1, 0.1]` إلى
الرموز الثلاثة، وبدّل الرمزين الأولين وأبقِ المواضع مكانها، فيتغيّر المخرَج. صار الترتيب مرئيًا.

</div>

In [ ]:
def positional_encoding(seq_len, d_model):
    """Sinusoidal positional encoding, shape (seq_len, d_model)."""
    # TODO: Build the angle matrix position / 10000^(2i/d), then fill even dimensions with sin and odd with cos.
    # مهمة: ابنِ مصفوفة الزوايا `position / 10000^(2i/d)`، ثم املأ الأبعاد الزوجية بـ`sin` والفردية بـ`cos`.


PE = positional_encoding(64, 32)
POSITION_IS_STABLE = torch.equal(positional_encoding(16, 32)[5], PE[5])
plt.imshow(PE.numpy(), aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
plt.xlabel("dimension")
plt.ylabel("position")
plt.title("sinusoidal positional encoding")
plt.colorbar()
plt.show()
print(f"position 5 is the same vector in a 16- and a 64-long sequence: {POSITION_IS_STABLE}")

In [ ]:
SLIDE_POSITIONS = np.array([[0.1, 0.0], [0.0, 0.1], [0.1, 0.1]])
swapped = V[[1, 0, 2]]                                   # swap the first two tokens, positions stay
plain = (attention(QUERY, V)[0], attention(QUERY, swapped)[0])
positioned = (attention(QUERY, V + SLIDE_POSITIONS)[0], attention(QUERY, swapped + SLIDE_POSITIONS)[0])
ORDER_VISIBLE = not np.allclose(*positioned)
print(f"without positions: {plain[0].round(4)} vs {plain[1].round(4)}  — identical")
print(f"with positions   : {positioned[0].round(4)} vs {positioned[1].round(4)}  — different: {ORDER_VISIBLE}")

### Task 2.3 — multi-head attention

Run several attentions side by side on slices of the vector, then join them. With `d_model = 8` and 2
heads, each head works in 4 dimensions. Every head gets its own query, key and value projections, so
each one can learn to look for something different. The helpers `split_heads` and `merge_heads` below
do the reshaping. You write the module.

Two checks. The constructor must **refuse** a `d_model` that does not divide by `n_heads`, with a
message that names both numbers — that error is the first one everyone hits. And with your weights
copied in, `nn.MultiheadAttention` must give the same attention weights to `1e-5`.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — الانتباه متعدّد الرؤوس

شغّل عدّة انتباهات جنبًا إلى جنب على شرائح من المتّجه، ثم اجمعها. حين `d_model = 8` ورأسان، يعمل كل رأس في ٤
أبعاد. ولكل رأس إسقاطاته الخاصة للمُستعلِم والمفتاح والقيمة، فيتعلّم كلٌّ منها البحث عن شيء مختلف. والدالّتان
المساعدتان `split_heads` و`merge_heads` أدناه تُعيدان التشكيل. وأنت تكتب الوحدة.

فحصان. يجب أن **يرفض** المُنشئ `d_model` لا يقبل القسمة على `n_heads`، برسالة تُسمّي العددين — فهذا أول خطأ
يصطدم به الجميع. وبأوزانك منسوخةً إليها، يجب أن تُعطي `nn.MultiheadAttention` أوزان الانتباه نفسها إلى `1e-5`.

</div>

In [ ]:
def split_heads(x, n_heads):
    """(batch, seq, d_model) -> (batch, n_heads, seq, d_head)."""
    batch, seq, d_model = x.shape
    return x.view(batch, seq, n_heads, d_model // n_heads).transpose(1, 2)


def merge_heads(x):
    """(batch, n_heads, seq, d_head) -> (batch, seq, d_model)."""
    batch, n_heads, seq, d_head = x.shape
    return x.transpose(1, 2).contiguous().view(batch, seq, n_heads * d_head)


class MultiHeadAttention(nn.Module):
    """Multi-head self-attention. Returns (output, per-head weights)."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        # TODO: Raise a ValueError naming both numbers if d_model % n_heads != 0, then build the four projections: query, key, value and out (each nn.Linear(d_model, d_model)).
        # مهمة: ارفع `ValueError` يُسمّي العددين إن كان `d_model % n_heads != 0`، ثم ابنِ الإسقاطات الأربعة: `query` و`key` و`value` و`out` (كلٌّ `nn.Linear(d_model, d_model)`).

    def forward(self, x, mask=None):
        # TODO: Project to q, k, v, split into heads, run scaled_attention (with the mask), merge, project out.
        # مهمة: أسقِط إلى `q` و`k` و`v`، واقسم إلى رؤوس، وشغّل `scaled_attention` (مع القناع)، واضمم، وأسقِط للخرج.

In [ ]:
torch.manual_seed(SEED)
X = torch.randn(1, 5, 8)                                       # (batch, seq, d_model)
mha = MultiHeadAttention(d_model=8, n_heads=2)
mha_output, mha_weights = mha(X)
print(f"input {tuple(X.shape)} -> weights {tuple(mha_weights.shape)} -> output {tuple(mha_output.shape)}")

try:
    MultiHeadAttention(8, 3)
    GUARD_FIRED = False
except ValueError as error:
    GUARD_FIRED = True
    print(f"\nthe guard, fired on purpose: {error}")

reference = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
with torch.no_grad():
    reference.in_proj_weight.copy_(torch.cat([mha.query.weight, mha.key.weight, mha.value.weight]))
    reference.in_proj_bias.copy_(torch.cat([mha.query.bias, mha.key.bias, mha.value.bias]))
    reference.out_proj.weight.copy_(mha.out.weight)
    reference.out_proj.bias.copy_(mha.out.bias)
_, reference_weights = reference(X, X, X, average_attn_weights=False)
WEIGHT_DIFFERENCE = float((mha_weights - reference_weights).abs().max())
print(f"\nlargest difference from nn.MultiheadAttention: {WEIGHT_DIFFERENCE:.1e}")

### Task 2.4 — the transformer block, and the causal mask

Around the attention go four more pieces, every one of them already familiar:

1. **attention**, then a **residual** connection (`x + attention(x)`) — ResNet's skip connection;
2. **LayerNorm**;
3. a **feed-forward** network applied to each token separately (`Linear → GELU → Linear`, 4× wider
   inside);
4. a second residual and a second LayerNorm.

Record the shape after every step. Each one must equal the input's shape — that is what lets you stack
blocks.

Then the **causal mask**: set every score above the diagonal to `−inf` before the softmax, so token `t`
can only attend to tokens `0 … t`. The weight matrix becomes lower-triangular, and every row still sums
to 1. It is the same idea as Wednesday's causal padding, and it is what a text generator — tomorrow's
LLM — runs on: without it, predicting the next token could just read it.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — كتلة المحوّل، والقناع السببي

حول الانتباه تأتي أربع قطع أخرى، وكلها مألوفة أصلًا:

١. **الانتباه**، ثم وصلة **متبقّية** (`x + attention(x)`) — وصلة التخطّي في ResNet؛
٢. **LayerNorm**؛
٣. شبكة **تغذية أمامية** تُطبَّق على كل رمز منفردًا (`Linear ← GELU ← Linear`، أعرض ٤ مرّات في الداخل)؛
٤. متبقّية ثانية وLayerNorm ثانية.

سجّل الشكل بعد كل خطوة. ويجب أن يساوي كلٌّ منها شكل المُدخل — فهذا ما يتيح لك تكديس الكتل.

ثم **القناع السببي**: اجعل كل درجة فوق القُطر `−inf` قبل softmax، فلا يستطيع الرمز `t` أن ينتبه إلا إلى الرموز
`0 … t`. فتصير مصفوفة الأوزان مثلّثية سفلية، ويبقى كل صف يجمع إلى ١. وهي فكرة الحشو السببي يوم الأربعاء نفسها، وعليها
يعمل مولِّد النصوص — نموذج اللغة الكبير غدًا: فبدونها يستطيع التنبّؤ بالرمز التالي أن يقرأه ببساطة.

</div>

In [ ]:
class TransformerBlock(nn.Module):
    """attention -> residual -> LayerNorm -> feed-forward -> residual -> LayerNorm."""

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.norm_1, self.norm_2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, mask=None):
        # TODO: Run the six steps, record each step's shape in `shapes`, and return (output, weights, shapes).
        # مهمة: شغّل الخطوات الستّ، وسجّل شكل كل خطوة في `shapes`، وأعِد `(output, weights, shapes)`.

In [ ]:
torch.manual_seed(SEED)
block = TransformerBlock(d_model=8, n_heads=2, d_ff=32)
block_output, _, BLOCK_SHAPES = block(X)
for step, shape in BLOCK_SHAPES.items():
    print(f"  {step:<20} {shape}")
PARAMETERS = sum(p.numel() for p in block.parameters())
print(f"\n{PARAMETERS:,} parameters in one block")

CAUSAL_MASK = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)   # True above the diagonal
_, masked_weights, _ = block(X, CAUSAL_MASK)
print("\ncausal weights, head 0:\n", masked_weights[0, 0].detach().numpy().round(2))
IS_LOWER_TRIANGULAR = bool((masked_weights[0, :, CAUSAL_MASK] == 0).all())
ROWS_SUM_TO_ONE = bool(torch.allclose(masked_weights.sum(-1), torch.ones(1), atol=1e-6))

### Task 2.5 — the memory task, second round

The same task, length 100: the length where Tuesday's recurrent cells were a lottery, and longer than
the 61-token receptive field of Wednesday's review TCN. The model: embed the tokens, **add the positional
encoding**, run **two** of your transformer blocks, and answer from the last position.

Train it with the same 800 steps (learning rate 0.003). It solves it. The last token's query can reach
position 0 directly, in one step of one layer — nothing has to survive a chain or fit in a window.

Then look at **how**. Plot the attention weights of the last position over the 100 positions, for each
layer, averaged over the heads. Somewhere in the stack, one sharp spike sits on position 0: the model
learned where the answer lives and looks there.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — مهمة الذاكرة، الجولة الثانية

المهمة نفسها بالطول ١٠٠: الطول الذي كانت فيه خلايا الثلاثاء المتكرّرة يانصيبًا، وهو أطول من مجال استقبال
TCN المراجعات يوم الأربعاء، ٦١ رمزًا. والنموذج: ضمِّن الرموز، و**أضِف ترميز الموضع**، وشغّل **كتلتين** من كتل المحوّل التي كتبتها،
وأجِب من الموضع الأخير.

درّبه بالخطوات نفسها، ٨٠٠ خطوة (معدّل تعلّم ٠٫٠٠٣). فيحلّها. فمُستعلِم الرمز الأخير يستطيع أن يصل إلى الموضع ٠
مباشرة، في خطوة واحدة من طبقة واحدة — لا شيء يجب أن ينجو من سلسلة أو يتّسع في نافذة.

ثم انظر **كيف**. ارسم أوزان انتباه الموضع الأخير على المواضع المئة، لكل طبقة، متوسّطةً على الرؤوس. ففي مكانٍ ما من
الكوم تجلس قمّة حادّة واحدة على الموضع ٠: تعلّم النموذج أين يسكن الجواب وصار ينظر هناك.

</div>

In [ ]:
# The memory task: the label is the first token; every token after it is random noise.
N_CLASSES, MEMORY_VOCAB = 4, 12          # tokens 0-3 can be labels, 4-11 are distractors


def memory_batch(n, length, generator):
    x = torch.randint(N_CLASSES, MEMORY_VOCAB, (n, length), generator=generator)
    y = torch.randint(0, N_CLASSES, (n,), generator=generator)
    x[:, 0] = y
    return x, y


def train_memory(model, length, steps=800, lr=0.01, seed=0):
    """Train on fresh random batches, then score 1,000 held-out sequences (chance is 0.25)."""
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = memory_batch(64, length, generator)
        loss = nn.functional.cross_entropy(model(x), y)
        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
    x, y = memory_batch(1000, length, torch.Generator().manual_seed(999))
    with torch.no_grad():
        return float((model(x).argmax(1) == y).float().mean())

In [ ]:
class TransformerMemory(nn.Module):
    def __init__(self, d_model=32, n_heads=2, n_blocks=2, max_len=128):
        super().__init__()
        self.embed = nn.Embedding(MEMORY_VOCAB, d_model)
        self.register_buffer("positions", positional_encoding(max_len, d_model))
        self.blocks = nn.ModuleList(TransformerBlock(d_model, n_heads, 4 * d_model) for _ in range(n_blocks))
        self.out = nn.Linear(d_model, N_CLASSES)

    def forward(self, x):
        # TODO: Embed, add the positional encoding for the first x.shape[1] positions, run every block (keep each block's weights in self.weights), and classify from the LAST position.
        # مهمة: ضمِّن، وأضِف ترميز الموضع لأول `x.shape[1]` موضعًا، وشغّل كل كتلة (واحفظ أوزان كلٍّ في `self.weights`)، وصنّف من الموضع الأخير.

In [ ]:
MEMORY_RESULTS = dict(json.loads(load_artefact("memory_task_D2.json").read_text(encoding="utf-8")))
start = time.perf_counter()
torch.manual_seed(0)
memory_transformer = TransformerMemory()
TRANSFORMER_MEMORY = train_memory(memory_transformer, 100, lr=0.003)
MEMORY_RESULTS["transformer @ 100 (seed 0)"] = TRANSFORMER_MEMORY
print(f"transformer, length 100: accuracy {TRANSFORMER_MEMORY:.3f}  ({time.perf_counter() - start:.0f} s)\n")
print(pd.Series(MEMORY_RESULTS).round(3).to_string())

In [ ]:
x, _ = memory_batch(256, 100, torch.Generator().manual_seed(7))
memory_transformer.eval()
with torch.no_grad():
    memory_transformer(x)
fig, axes = plt.subplots(1, len(memory_transformer.weights), figsize=(11, 3), sharey=True)
LAST_ROW = []
for layer, (ax, weights) in enumerate(zip(axes, memory_transformer.weights)):
    last_row = weights[:, :, -1, :].mean(dim=(0, 1))          # last query, averaged over heads and samples
    LAST_ROW.append(last_row)
    ax.bar(range(100), last_row.numpy())
    ax.set_title(f"block {layer + 1}: where the last position looks")
    ax.set_xlabel("position")
plt.show()
PEAK_ON_FIRST = max(float(row[0]) for row in LAST_ROW)
print(f"largest weight on position 0 in any block: {PEAK_ON_FIRST:.2f} (uniform would be 0.01)")

## Section 3 — Stretch: a transformer on the reviews  (≈30 min)

Put the block to work on the week's task. The classifier: embedding → positional encoding → two
transformer blocks (`d_model = 64`, 4 heads) → the mean of the real positions (masking out the
padding) → `Linear`. Train for three epochs on the week's split, add the leaderboard row, and run both
sentence pairs.

Expect it **not** to beat Tuesday's GRU or Wednesday's TCN by much — and perhaps to lose. A
transformer trained from scratch on 9,600 reviews has no advantage over them; its strengths are
reaching any distance in one step, and training in parallel on enormous data. Tomorrow's model has
exactly that: the same blocks, pretrained on more text than you could read in many lifetimes.

<div dir="rtl" align="right">

## القسم الثالث — إضافي: محوّل على المراجعات (نحو ٣٠ دقيقة)

شغّل الكتلة على مهمة الأسبوع. المُصنّف: تضمين ← ترميز الموضع ← كتلتا محوّل (`d_model = 64`، ٤ رؤوس) ← متوسط
المواضع الحقيقية (مع إخفاء الحشو) ← `Linear`. درّبه ثلاث حقب على تقسيم الأسبوع، وأضِف صفّ اللوحة، وشغّل زوجَي
الجمل.

وتوقّع **ألّا** يتفوّق كثيرًا على GRU الثلاثاء أو TCN الأربعاء — وقد يخسر. فالمحوّل المُدرَّب من الصفر على ٩٬٦٠٠
مراجعة لا ميزة له عليهما؛ وقوّته في بلوغ أي مسافة بخطوة واحدة، وفي التدريب على التوازي على بيانات هائلة. ونموذج الغد
عنده ذلك بالضبط: الكتل نفسها، مُدرَّبةً مسبقًا على نصٍّ أكثر مما تستطيع قراءته في أعمار كثيرة.

</div>

In [ ]:
MAX_LEN, VOCAB_SIZE = 128, 20_000
NEGATION_PAIR = ["the service was good", "the service was not good"]
ORDER_PAIR = ["the food was good but the service was bad",
              "the service was good but the food was bad"]
reviews = pd.read_parquet(get_dataset("reviews_sentiment"))
split = json.loads(load_artefact("week5_split.json").read_text(encoding="utf-8"))
check(hashlib.sha256(json.dumps({"train": split["train"], "test": split["test"]}).encode()).hexdigest()
      == split["sha256"], "week5_split.json must match its own hash", "يجب أن يطابق `week5_split.json` بصمته")
train, test = reviews.iloc[split["train"]], reviews.iloc[split["test"]]
print(f"train {len(train):,} | test {len(test):,} | split {split['sha256'][:12]}…")

def tokenize(text):
    return re.sub(r"[^0-9a-z؀-ۿ\s']+", " ", text.lower()).split()


counts = collections.Counter(w for text in train.review_text for w in tokenize(text))
ITOS = ["<pad>", "<unk>"] + [w for w, _ in counts.most_common(VOCAB_SIZE - 2)]
STOI = {w: i for i, w in enumerate(ITOS)}


def encode(texts):
    """Token ids, right-padded with 0 to MAX_LEN, and the true length of each review."""
    ids = [[STOI.get(w, 1) for w in tokenize(t)][:MAX_LEN] or [1] for t in texts]
    x = torch.zeros(len(ids), MAX_LEN, dtype=torch.long)
    for i, seq in enumerate(ids):
        x[i, :len(seq)] = torch.tensor(seq)
    return x, torch.tensor([len(seq) for seq in ids])


X_train, len_train = encode(train.review_text)
X_test, len_test = encode(test.review_text)
y_train = torch.tensor(train.sentiment.to_numpy()).long()
y_test = torch.tensor(test.sentiment.to_numpy()).long()
print(f"vocabulary {len(ITOS):,} | X_train {tuple(X_train.shape)} | longest review {int(len_train.max())} tokens")

def load_board(days):
    """The leaderboard rows earlier days wrote (each day keeps its own file)."""
    rows = []
    for day in days:
        rows += json.loads(load_artefact(f"leaderboard_{day}.json").read_text(encoding="utf-8"))
    return rows


def show_board(rows):
    columns = ["day", "model", "n_train", "accuracy", "f1", "fit_s", "ms_per_1000", "size_mb",
               "negation_ok", "order_gap"]
    print(pd.DataFrame(rows)[columns].to_string(index=False))

In [ ]:
class ReviewTransformer(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_blocks=2):
        super().__init__()
        self.embed = nn.Embedding(len(ITOS), d_model, padding_idx=0)
        self.register_buffer("positions", positional_encoding(MAX_LEN, d_model))
        self.blocks = nn.ModuleList(TransformerBlock(d_model, n_heads, 4 * d_model) for _ in range(n_blocks))
        self.out = nn.Linear(d_model, 2)

    def forward(self, x, lengths):
        padded = torch.arange(x.shape[1])[None, :] >= lengths[:, None]       # (batch, seq), True on padding
        h = self.embed(x) + self.positions[:x.shape[1]]
        for block in self.blocks:
            h, _, _ = block(h, padded[:, None, None, :])                     # no query may look at padding
        h = h.masked_fill(padded[:, :, None], 0.0)
        return self.out(h.sum(1) / lengths[:, None])


def accuracy(model):
    model.eval()
    with torch.no_grad():
        return float((model(X_test, len_test).argmax(1) == y_test).float().mean())


from sklearn.metrics import f1_score

torch.manual_seed(SEED)
review_transformer = ReviewTransformer()
optimiser = torch.optim.Adam(review_transformer.parameters(), lr=1e-3)
start = time.perf_counter()
for epoch in range(3):
    review_transformer.train()
    order = torch.randperm(len(X_train))
    for i in range(0, len(order), 64):
        batch = order[i:i + 64]
        loss = nn.functional.cross_entropy(review_transformer(X_train[batch], len_train[batch]), y_train[batch])
        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(review_transformer.parameters(), 1.0)
        optimiser.step()
    print(f"  epoch {epoch + 1}: test accuracy {accuracy(review_transformer):.4f}")
TRANSFORMER_FIT_S = time.perf_counter() - start

In [ ]:
review_transformer.eval()
start = time.perf_counter()
with torch.no_grad():
    transformer_predictions = review_transformer(X_test, len_test).argmax(1).numpy()
    TRANSFORMER_MS_PER_1000 = (time.perf_counter() - start) / len(X_test) * 1e6
    x_pairs, l_pairs = encode(NEGATION_PAIR + ORDER_PAIR)
    p_pairs = torch.softmax(review_transformer(x_pairs, l_pairs), 1)[:, 1].numpy()

BOARD_D4 = [{
    "day": "D4", "model": "transformer (trained from scratch)", "n_train": len(X_train),
    "accuracy": round(accuracy(review_transformer), 4), "f1": round(f1_score(y_test.numpy(), transformer_predictions), 4),
    "fit_s": round(TRANSFORMER_FIT_S, 1), "ms_per_1000": round(TRANSFORMER_MS_PER_1000, 1),
    "size_mb": round(sum(p.numel() for p in review_transformer.parameters()) * 4 / 1e6, 2),
    "negation_ok": bool(p_pairs[0] > 0.5 > p_pairs[1]), "order_gap": round(float(abs(p_pairs[2] - p_pairs[3])), 4),
}]
for sentence, p in zip(NEGATION_PAIR + ORDER_PAIR, p_pairs):
    print(f"  p(positive) = {p:.4f}   {sentence}")
print()
show_board(load_board(["D1", "D2", "D3"]) + BOARD_D4)

## Save your artefacts

- `block_shapes.json` — the shape after every step of the block, its parameter count, and the checks.
- `memory_task_D4.json` — the complete memory table: RNN, GRU, LSTM and the transformer.
- `leaderboard_D4.json` — the transformer's row, if you ran the stretch.

<div dir="rtl" align="right">

## احفظ آثارك

- `block_shapes.json` — الشكل بعد كل خطوة من الكتلة، وعدد معاملاتها، والفحوص.
- `memory_task_D4.json` — جدول الذاكرة الكامل: RNN وGRU وLSTM والمحوّل.
- `leaderboard_D4.json` — صفّ المحوّل، إن شغّلت القسم الإضافي.

</div>

In [ ]:
(ARTEFACT_DIR / "block_shapes.json").write_text(json.dumps({
    "d_model": 8, "n_heads": 2, "d_ff": 32, "parameters": PARAMETERS,
    "shapes": {step: list(shape) for step, shape in BLOCK_SHAPES.items()},
    "matches_torch_multihead": WEIGHT_DIFFERENCE < 1e-5,
    "causal_mask": {"lower_triangular": IS_LOWER_TRIANGULAR, "rows_sum_to_one": ROWS_SUM_TO_ONE},
}, indent=2), encoding="utf-8")
(ARTEFACT_DIR / "memory_task_D4.json").write_text(json.dumps(MEMORY_RESULTS, indent=2), encoding="utf-8")
if "BOARD_D4" in globals():
    (ARTEFACT_DIR / "leaderboard_D4.json").write_text(json.dumps(BOARD_D4, indent=2), encoding="utf-8")
print("wrote block_shapes.json, memory_task_D4.json" + (" and leaderboard_D4.json" if "BOARD_D4" in globals() else ""))

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(np.allclose(WEIGHTS, [0.42, 0.16, 0.42], atol=0.005) and np.allclose(OUTPUT, [0.84, 0.58], atol=0.005)
      and np.allclose(FLIPPED_WEIGHTS, [0.16, 0.42, 0.42], atol=0.005),
      "attention must reproduce the slide: weights 0.42/0.16/0.42, output [0.84, 0.58], flipped 0.16/0.42/0.42",
      "يجب أن يُعيد الانتباه إنتاج الشريحة: الأوزان ٠٫٤٢/٠٫١٦/٠٫٤٢، والمخرَج `[0.84, 0.58]`، والمقلوبة ٠٫١٦/٠٫٤٢/٠٫٤٢")
check(ORDER_BLIND and ORDER_VISIBLE,
      "reversing the tokens must leave plain attention unchanged, and positions must make a swap visible",
      "يجب ألّا يُغيّر عكس الرموز الانتباه المجرّد، وأن تجعل المواضع التبديل مرئيًا")
check(SLIDE_MATCH and SCALED.max() < UNSCALED.max() and POSITION_IS_STABLE,
      "batched attention must match the warm-up, sqrt(d) must soften the softmax, and a position's encoding must not depend on the length",
      "يجب أن يطابق الانتباه المُجمَّع الإحماء، وأن تُليّن `sqrt(d)` دالّة softmax، وألّا يعتمد ترميز الموضع على الطول")
check(GUARD_FIRED and WEIGHT_DIFFERENCE < 1e-5,
      f"n_heads=3 must raise your ValueError, and your weights must match nn.MultiheadAttention to 1e-5 (got {WEIGHT_DIFFERENCE:.1e})",
      f"يجب أن يرفع `n_heads=3` خطأك، وأن تطابق أوزانك `nn.MultiheadAttention` إلى ١e−٥ (والناتج {WEIGHT_DIFFERENCE:.1e})")
check(all(shape == BLOCK_SHAPES["input"] for shape in BLOCK_SHAPES.values()),
      f"every step of the block must keep the input's shape — got {BLOCK_SHAPES}",
      f"يجب أن تُبقي كل خطوة في الكتلة شكل المُدخل — والناتج {BLOCK_SHAPES}")
check(IS_LOWER_TRIANGULAR and ROWS_SUM_TO_ONE,
      "the causal weights must be zero above the diagonal, with every row still summing to 1",
      "يجب أن تكون الأوزان السببية صفرًا فوق القُطر، وأن يبقى كل صف يجمع إلى ١")
check(TRANSFORMER_MEMORY > 0.9,
      f"the transformer must solve the length-100 memory task — got {TRANSFORMER_MEMORY:.3f}",
      f"يجب أن يحلّ المحوّل مهمة الذاكرة بالطول ١٠٠ — والناتج {TRANSFORMER_MEMORY:.3f}")
report()

## What's next

**W5D5 — LLMs.** Today you built the block and trained tiny ones from scratch. On the reviews, the
transformer did not beat the GRU or the TCN: architecture alone is not enough.

Tomorrow you run a model built from these same blocks — decoder blocks, with today's causal mask —
that was **pretrained** on trillions of tokens to do one thing: predict the next token. You will look
inside its next-token probabilities, write the sampling that turns them into text, and use it to
classify the week's reviews **without training it at all**. Then the leaderboard is complete, and you
decide which model you would ship.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٥ اليوم ٥ — نماذج اللغة الكبيرة.** بنيت اليوم الكتلة ودرّبت كتلًا صغيرة من الصفر. وعلى المراجعات لم
يتفوّق المحوّل على GRU ولا على TCN: فالمعمارية وحدها لا تكفي.

وغدًا تُشغّل نموذجًا مبنيًا من هذه الكتل نفسها — كتل مُفكِّك، بقناع اليوم السببي — **دُرِّب مسبقًا** على تريليونات
الرموز لشيء واحد: التنبّؤ بالرمز التالي. ستنظر داخل احتمالاته للرمز التالي، وتكتب أخذ العيّنات الذي يحوّلها إلى
نص، وتستعمله لتصنيف مراجعات الأسبوع **دون أي تدريب**. ثم تكتمل اللوحة، وتقرّر أيّ نموذج كنت لتنشره.

</div>